# Radio Celendín AutoRadio: análisis con Spark
**Datos sintéticos, no producidos por la emisora.** Ejecutar en Jupyter o Colab, primero generar `dashboard_datos.csv`. Capturar salida y tiempo para la EF.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from time import perf_counter
spark = SparkSession.builder.master("local[*]").appName("Autoradio_EF").getOrCreate()
print("Spark:",spark.version)
path = Path("../03_bi/dashboard_datos.csv")
if not path.exists():
    path = Path("dashboard_datos.csv")  # Colab: subir el archivo a la sesión
assert path.exists(), "Sube el CSV del repositorio si usas Colab"
df = spark.read.option("header",True).option("inferSchema",True).csv(str(path))
df.printSchema(); print("Total:",df.count())

In [ ]:
# Objeto 1: DataFrame; agregar por campaña
kpi=df.groupBy("codigo_campana").agg(F.count("*").alias("programados"),F.sum("emitidos").alias("emitidos"))
kpi.orderBy("codigo_campana").show(20)

In [ ]:
# Objeto 2: Spark SQL; misma pregunta expresada en SQL
df.createOrReplaceTempView("emisiones")
spark.sql("SELECT codigo_campana, COUNT(*) AS programados, SUM(emitidos) AS emitidos FROM emisiones GROUP BY codigo_campana ORDER BY codigo_campana").show()

In [ ]:
# Objeto 3: RDD (comparación didáctica, no recomendado para tablas de producción)
rdd = df.select("codigo_campana","emitidos").rdd.map(lambda r:(r[0],int(r[1])))
print(sorted(rdd.reduceByKey(lambda a,b:a+b).collect()))

In [ ]:
# Estadística descriptiva de retrasos
from pyspark.sql.functions import stddev, avg
(df.where(F.col("atraso_seg").isNotNull())
 .agg(avg("atraso_seg").alias("media_seg"),stddev("atraso_seg").alias("desviacion_seg"))
 .show())

In [ ]:
# Benchmark local comparable con 02_benchmark_sqlserver.py.
# NOTA: Spark tiene overhead fijo; comparar sólo tiempos tomados en hardware semejante.
for n in [100,500,960]:
    part = df.limit(n).cache();part.count()
    t0=perf_counter()
    r=part.groupBy('codigo_campana').agg(F.sum('emitidos')).collect()
    print('Filas',n,'Tiempo Spark (s)',round(perf_counter()-t0,6),'grupos',len(r))
    part.unpersist()
print('Agregar aquí los tiempos SQL Server de 02_benchmark_sqlserver.py y discutirlos.');